# CDR-MLC causal adaptive router experiment

This notebook tests whether causal, label-free centroid adaptation repairs routing collapse. The original feature window remains unchanged: `SynAck`, `AckDat`, and `TcpRtt` are converted to 15 causal statistics using a window of 3.

For every block, prediction is performed **before** the block is added to the adaptation buffer. Only past unlabeled 15-dimensional routing vectors can affect the next block. Experts and the training scaler remain frozen.

This is an exploratory router-only experiment, not a replacement for the main results.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

try:
    display
except NameError:
    display = print

RANDOM_STATE = 42
BASE = Path('DATASETS/CDR-MLC/scale_1')
SCENARIOS = {
    'scenario_4': (BASE/'Short/CDR-MLC-Shuffle.csv', BASE/'Long/CDR-MLC-Shuffle.csv'),
    'scenario_5': (BASE/'Long/CDR-MLC-Shuffle.csv', BASE/'Short/CDR-MLC-Shuffle.csv'),
}

main_notebook = json.loads(Path('CDR-MLC.ipynb').read_text(encoding='utf-8'))
exec(compile(''.join(main_notebook['cells'][0]['source']), 'CDR-MLC.ipynb::core', 'exec'), globals())
print('Loaded the exact leakage-safe CDR-MLC implementation')


In [ ]:
def _predict_by_routes(frame, features, routes, classifiers):
    predictions = np.zeros(len(frame), dtype=int)
    for cluster_id, classifier in classifiers.items():
        mask = routes == cluster_id
        if mask.any():
            predictions[mask] = classifier.predict(frame.loc[mask, features])
    return predictions


def run_causal_adaptive_router(
    scenario,
    block_size=256,
    adaptation_buffer_size=2048,
    update_every=2048,
    centroid_learning_rate=0.20,
    maximum_step_radius=0.50,
):
    train_path, test_path = SCENARIOS[scenario]
    result = run_pipeline_from_two_files(
        train_file=str(train_path), test_file=str(test_path),
        n_clusters=3, window_size=3,
        clustering_stats=['mean','median','std','min','max']
    )
    train_df, test_df = result['train_df'], result['test_df']
    features = result['classification_features']

    train_stats, _ = compute_sliding_window_stats(
        train_df, result['used_fixed_features'], result['window_size'], result['clustering_stats'])
    test_stats, _ = compute_sliding_window_stats(
        test_df, result['used_fixed_features'], result['window_size'], result['clustering_stats'])
    train_routing = result['scaler'].transform(train_stats)
    test_routing = result['scaler'].transform(test_stats)

    initial_centers = result['kmeans_model'].cluster_centers_.copy()
    centers = initial_centers.copy()
    train_routes = train_df['cluster'].to_numpy()
    cluster_radius = np.array([
        np.quantile(np.linalg.norm(train_routing[train_routes == k] - initial_centers[k], axis=1), .95)
        for k in range(3)
    ])

    adaptive_routes = np.zeros(len(test_df), dtype=int)
    adaptive_predictions = np.zeros(len(test_df), dtype=int)
    history, update_log = [], []

    for start in range(0, len(test_df), block_size):
        end = min(start + block_size, len(test_df))
        current = test_routing[start:end]

        # Strict order: predict this block with the state learned only from the past.
        routes = np.linalg.norm(current[:, None, :] - centers[None, :, :], axis=2).argmin(axis=1)
        adaptive_routes[start:end] = routes
        block_frame = test_df.iloc[start:end]
        for cluster_id, classifier in result['classifiers'].items():
            mask = routes == cluster_id
            if mask.any():
                adaptive_predictions[start:end][mask] = classifier.predict(block_frame.loc[mask, features])

        # The already-predicted unlabeled block may now influence future blocks.
        history.append(current)
        while sum(len(part) for part in history) > adaptation_buffer_size:
            history.pop(0)

        if end >= adaptation_buffer_size and end % update_every < block_size:
            past = np.vstack(history)
            candidate = KMeans(
                n_clusters=3, init=centers, n_init=1, max_iter=30,
                random_state=RANDOM_STATE
            ).fit(past).cluster_centers_

            old_ids, new_ids = linear_sum_assignment(
                np.linalg.norm(centers[:, None, :] - candidate[None, :, :], axis=2))
            ordered = np.empty_like(candidate)
            for old_id, new_id in zip(old_ids, new_ids):
                ordered[old_id] = candidate[new_id]

            delta = ordered - centers
            for cluster_id in range(3):
                norm = np.linalg.norm(delta[cluster_id])
                limit = maximum_step_radius * cluster_radius[cluster_id]
                if norm > limit and norm > 0:
                    delta[cluster_id] *= limit / norm
            centers += centroid_learning_rate * delta
            update_log.append({'after_sample': end, 'center_movement': np.linalg.norm(centers-initial_centers, axis=1).copy()})

    y_true = test_df[result['target_column']].to_numpy()
    frozen_routes = test_df['cluster'].to_numpy()
    frozen = result['test_results']
    adaptive = {
        'accuracy': accuracy_score(y_true, adaptive_predictions),
        'f1_weighted': f1_score(y_true, adaptive_predictions, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_true, adaptive_predictions, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, adaptive_predictions, average='weighted', zero_division=0),
    }
    comparison = pd.DataFrame([
        {'router':'Frozen K-Means', 'accuracy':frozen['accuracy'], 'f1_weighted':frozen['f1_weighted'],
         'route_0':np.sum(frozen_routes==0), 'route_1':np.sum(frozen_routes==1), 'route_2':np.sum(frozen_routes==2)},
        {'router':'Causal adaptive K-Means', **adaptive,
         'route_0':np.sum(adaptive_routes==0), 'route_1':np.sum(adaptive_routes==1), 'route_2':np.sum(adaptive_routes==2)},
    ])
    print('\nFrozen versus causal adaptive router')
    display(comparison.round(4))
    print('Final centroid movement:', np.linalg.norm(centers-initial_centers, axis=1))
    print('Number of causal updates:', len(update_log))

    comparison.set_index('router')[['route_0','route_1','route_2']].plot.bar(
        figsize=(8,4), rot=0, title=f'{scenario}: routing counts')
    plt.ylabel('Samples'); plt.grid(axis='y', alpha=.25); plt.tight_layout(); plt.show()
    return {'base_result':result, 'comparison':comparison, 'adaptive_routes':adaptive_routes,
            'adaptive_predictions':adaptive_predictions, 'initial_centers':initial_centers,
            'final_centers':centers, 'update_log':update_log, 'parameters':{
                'block_size':block_size, 'adaptation_buffer_size':adaptation_buffer_size,
                'update_every':update_every, 'centroid_learning_rate':centroid_learning_rate,
                'maximum_step_radius':maximum_step_radius}}


In [ ]:
# Scenario 4: Short -> Long
scenario_4_adaptive = run_causal_adaptive_router('scenario_4')


In [ ]:
# Scenario 5: Long -> Short
scenario_5_adaptive = run_causal_adaptive_router('scenario_5')
